# 06. TourAPI, Cache, Fallback, Context 학습 흐름

검증 목적: 각 셀을 위에서 아래로 실행하며 원본형 앱의 구조와 계약을 직접 확인합니다.

이 노트북은 NewNote 강의 노트처럼 설명 셀과 코드 셀을 번갈아 배치합니다. 목표는 `관광 API 키가 없어도 학습 가능한 캐시/폴백/지역/의도/카드 근거 흐름을 확인합니다.` 입니다. 관련 장은 08 TourAPI, 09 Cache/Fallback, 10 Parsing, 11 Intent/Context, 12 Card Evidence 입니다.

## 실행 전 준비

- 저장소 루트에서 Jupyter 커널을 시작합니다.
- 긴 서버를 백그라운드로 띄우지 않고, 가능한 한 TestClient와 파일 읽기로 확인합니다.
- 개인 `.env` 값, API 키, 로컬 DB 경로는 출력하지 않습니다.
- 이번 노트북의 초점: 이 노트북은 원본 앱 표면의 핵심인 관광 옵션 흐름을 가장 자세히 다룹니다.

In [ ]:
# 공통 경로 셀
# 모든 노트북은 저장소 루트에서 실행한다고 가정합니다.
from pathlib import Path
PROJECT_ROOT = Path.cwd()
TEMPLATE_ROOT = PROJECT_ROOT / 'project_template'
print('PROJECT_ROOT:', PROJECT_ROOT.name)
print('TEMPLATE_ROOT exists:', TEMPLATE_ROOT.exists())
assert TEMPLATE_ROOT.exists(), 'project_template 폴더가 보여야 합니다.'

## 원본형 앱 연결

아래 셀부터는 작은 실험을 통해 원본형 앱의 어느 파일과 연결되는지 확인합니다. 코드가 길어 보여도 목적은 하나입니다. 초급자는 출력된 값이 기대와 다른 순간 바로 이전 셀로 돌아가면 됩니다.

### 1. 서비스 파일 맵

이 셀은 `서비스 파일 맵`을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 사용합니다.

In [ ]:
from pathlib import Path
ROOT = Path.cwd()
SERVICE_ROOT = ROOT / 'project_template' / 'app' / 'services'
required = ['tour_api_service.py', 'tourism_intent_classifier.py', 'tourism_context_classifier.py', 'tourism_card_codec.py', 'tourism_query_service.py']
for name in required:
    path = SERVICE_ROOT / name
    print(name, path.exists())
    assert path.exists(), name

### 2. 지역 코드 데이터 확인

이 셀은 `지역 코드 데이터 확인`을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 사용합니다.

In [ ]:
import json
processed = ROOT / 'project_template' / 'data' / 'processed'
for name in ['tour_area_codes.json', 'tourapi_bigdata_region_codes.json']:
    data = json.loads((processed / name).read_text(encoding='utf-8'))
    print(name, type(data).__name__, str(data)[:300])
    assert data

### 3. 간단 지역 파싱 실험

이 셀은 `간단 지역 파싱 실험`을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 사용합니다.

In [ ]:
query = '부산 해운대에서 비 오는 날 갈 만한 곳'
regions = ['서울', '부산', '제주', '경주', '해운대']
matched = [region for region in regions if region in query]
print(matched)
assert '부산' in matched

### 4. 의도 분류 입력 관찰

이 셀은 `의도 분류 입력 관찰`을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 사용합니다.

In [ ]:
examples = ['아이와 갈 곳', '휠체어 접근 가능한 곳', '실내 관광지', '방금 말한 곳 근처 맛집']
for text in examples:
    labels = []
    if '휠체어' in text:
        labels.append('accessibility')
    if '아이' in text:
        labels.append('family')
    if '방금' in text:
        labels.append('context_followup')
    print(text, labels or ['general'])

### 5. FastAPI 관광 path smoke

이 셀은 `FastAPI 관광 path smoke`을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 사용합니다.

In [ ]:
import sys
from fastapi.testclient import TestClient
TEMPLATE_ROOT = ROOT / 'project_template'
sys.path.insert(0, str(TEMPLATE_ROOT))
from app.main import app
client = TestClient(app)
paths = sorted(path for path in client.get('/openapi.json').json().get('paths', {}) if 'tour' in path.lower())
print(paths)
assert paths

### 6. 카드 근거 정책 요약

이 셀은 `카드 근거 정책 요약`을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 사용합니다.

In [ ]:
policy = {
    'accessibility': 'raw_fields_only',
    'fallback': 'must mark source',
    'cache': 'do not expose secret key',
}
for key, value in policy.items():
    print(key, '=>', value)
assert policy['accessibility'] == 'raw_fields_only'

### 7. 후속 질문 세션 메모

이 셀은 `후속 질문 세션 메모`을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 사용합니다.

In [ ]:
session_notes = []
session_notes.append({'turn': 1, 'region': '부산', 'condition': 'wheelchair'})
session_notes.append({'turn': 2, 'question': '그중 실내 위주로', 'inherits_region': True})
print(session_notes)
assert session_notes[-1]['inherits_region']

## 정리

이 노트북에서 본 것은 최종 앱 전체가 아니라, 한 장의 핵심 계약입니다. 같은 원리가 `project_template/app`, `project_template/frontend`, `project_template/data` 안의 실제 파일로 정리되어 있습니다.

In [ ]:
summary = {
    'notebook': 'completed',
    'next_step': '관련 chapter 문서를 읽고 같은 검증을 테스트로 반복합니다.',
}
print(summary)
assert summary['notebook'] == 'completed'